# OpenCode: AI Coding Agent in the Terminal


OpenCode is a **terminal UI** (TUI) that gives an AI coding agent full access to the filesystem, shell, and git — running directly from the command line without a browser or IDE plugin. It is provider-agnostic, supporting any LLM backend through a unified configuration layer. This notebook documents the project's OpenCode setup for `ai-notebooks` specifically: how we install and connect it, how we configure project context, and how the custom agents and skills in `.opencode/` extend its default behavior.


## Installation & Setup


**Install.** We install OpenCode via the official install script or via Homebrew:

```{.bash filename="$ (local)"}
curl -fsSL https://opencode.ai/install | bash
```

Alternatively:

```{.bash filename="$ (local)"}
brew install anomalyco/tap/opencode
```


**Connect.** We run `/connect` inside an OpenCode session to open the provider selection TUI, where we choose an LLM provider (e.g. Anthropic, OpenAI, Google) and a specific model. The resulting configuration is written to `~/.config/opencode/opencode.json` and is global — it is not part of any individual project.


**Initialize.** We run `/init` once per project. OpenCode scans the repository structure and writes an `AGENTS.md` file at the project root with a summary of the codebase, build commands, and conventions. This file is loaded as context at the start of every session.

:::{.callout-tip}
Commit `AGENTS.md` to the repo. It becomes permanent project context loaded on every session, and co-contributors benefit from the same baseline understanding.
:::


## Basic Usage


**Agents.** Two primary agents are cycled with **Tab**: **Build** (the default, which can edit files and run shell commands) and **Plan** (restricted, prompts before any edits or bash commands). The recommended workflow is to use Plan to reason through a problem first — exploring the codebase, identifying constraints, sketching a solution — then switch to Build to execute.


**File references.** We type `@` in the input field to open a fuzzy-search picker over the project files. One or more files can be attached to the context window this way, directing the agent's attention to specific modules or notebooks before asking a question.


**Action palette.** `Ctrl+P` opens a command palette listing all available actions, including model switching, history navigation, and session management. This is the quickest way to discover functionality without leaving the TUI.


**Session utilities.** `/share` creates a shareable link to the current conversation — useful for reproducing a result or filing a bug report. `/undo` and `/redo` revert or replay the most recent file-editing action, providing a lightweight safety net when the agent makes an unwanted change.


## Commands


OpenCode ships with a set of built-in slash commands for session management and project setup:

| Command | Description |
| :-- | :-- |
| `/init` | Scans project and writes `AGENTS.md` with project context |
| `/connect` | Opens TUI to select LLM provider and model |
| `/share` | Creates a shareable link to the current session |
| `/undo` | Reverts the last file-editing action |
| `/redo` | Replays the last undone file-editing action |
| `/help` | Shows available commands and keybindings |


Beyond the built-ins, we can define **custom commands** stored as markdown files under `.opencode/commands/<name>.md` and invoked as `/<name>` or `/<name> <arguments>`. Three interpolation mechanisms are available: `$ARGUMENTS` expands to the argument string passed at invocation, `` !`cmd` `` runs a shell command and interpolates its stdout, and `@file/path` reads and interpolates file contents.


For example, a `/review` command for style checking might live at `.opencode/commands/review.md`:

```markdown
Review the following file for style and correctness:

@$ARGUMENTS
```

Invoking `/review notebooks/deep/01.ipynb` causes OpenCode to read that file and pass it to the agent along with the review instruction.


## Project Context


**opencode.json.** The project-level config lives at the repository root. Ours is minimal:

```json
{
  "$schema": "https://opencode.ai/config.json",
  "instructions": [".opencode/preferences.md", ".github/AI_PREFERENCES.md"]
}
```

The `instructions` field takes a list of markdown file paths (or glob patterns) whose contents are prepended to every session's context window — in our case, the notebook output reading policy and cell reference notation.

:::{.callout-note}
`opencode.json` belongs at the **project root**, not inside `.opencode/`. The docs confirm this explicitly: "Place project specific config in the root of your project." The `.opencode/` directory is reserved for agents, commands, plugins, skills, and tools — not the main config file.
:::

:::{.callout-tip}
The config supports `{env:VAR_NAME}` and `{file:path}` syntax for injecting environment variables or file contents. This is useful for personalizing context per-user (e.g. `{env:OPENCODE_USER_NOTES}`) without committing private content to the repo.
:::


**AGENTS.md.** The primary project context file, generated by `/init` and maintained manually thereafter. It documents project structure, build commands (`make venv`, `make docs`, `ruff check .`), testing philosophy, and notebook writing conventions. OpenCode reads it at session start unconditionally.


**Global vs project config.** The split is deliberate: `~/.config/opencode/opencode.json` holds provider credentials and model preferences and is never committed; `opencode.json` is project-specific and belongs in version control alongside the code it describes.


## Permissions


By default OpenCode allows all operations without asking. The `permission` config key controls this. Each rule resolves to one of three actions: `"allow"` (run silently), `"ask"` (prompt for approval), or `"deny"` (block entirely).

The simplest form sets all tools at once:

```json
{
  "permission": "ask"
}
```

Or we can set a global default and override specific tools:

```json
{
  "permission": {
    "*": "ask",
    "bash": "allow",
    "edit": "deny"
  }
}
```


**Granular rules.** For `bash` and `edit`, the permission value can be an object mapping command patterns to actions rather than a flat string. Rules are evaluated in order and the **last matching rule wins** — so we put the catch-all `"*"` first and more specific rules after:

```json
{
  "permission": {
    "bash": {
      "*": "ask",
      "git *": "allow",
      "grep *": "allow",
      "rm *": "deny"
    },
    "edit": {
      "*": "deny",
      "notebooks/**": "allow"
    }
  }
}
```

Patterns follow simple glob syntax: `*` matches zero or more characters, `?` matches exactly one.


**This project's bash permissions.** We configure `~/.config/opencode/opencode.json` to allow most commands silently but require approval for destructive or irreversible operations:

```json
{
  "permission": {
    "bash": {
      "*": "allow",
      "rm *": "ask",
      "rm -rf *": "ask",
      "git push *": "ask",
      "git push --force *": "deny",
      "git reset --hard *": "ask",
      "sudo *": "ask",
      "curl * | *": "ask",
      "wget * | *": "ask"
    }
  }
}
```

The rules work as follows — since last matching rule wins, the specific patterns after `"*": "allow"` take precedence:

| Rule | Action | Reason |
| :-- | :--: | :-- |
| `*` | `allow` | Everything not matched below runs silently |
| `rm *` | `ask` | File deletion requires approval |
| `rm -rf *` | `ask` | Recursive deletion requires approval |
| `git push *` | `ask` | Confirm before pushing to remote |
| `git push --force *` | `deny` | Force push blocked entirely |
| `git reset --hard *` | `ask` | Destructive reset requires approval |
| `sudo *` | `ask` | Privilege escalation requires approval |
| `curl * \| *` | `ask` | Piped install scripts require approval |
| `wget * \| *` | `ask` | Piped install scripts require approval |

Force push is the only hard `deny` — it is almost never recoverable without coordinator intervention and there is no legitimate automated use case for it here.

:::{.callout-caution}
The `ask` interaction offers three outcomes: `once`, `always` (for the rest of the session), or `reject`. Choosing `always` incrementally builds up a set of trusted patterns without needing them hard-coded upfront — but be mindful that `always` persists for the entire session, not just one conversation turn.
:::


**Available permissions.** The full set of permissioned tools:

| Key | Matches against |
| :-- | :-- |
| `read` | File path |
| `edit` | File path (covers `edit`, `write`, `patch`, `multiedit`) |
| `glob` | Glob pattern |
| `grep` | Regex pattern |
| `list` | Directory path |
| `bash` | Parsed command string |
| `task` | Subagent type |
| `skill` | Skill name |
| `webfetch` | URL |
| `external_directory` | Paths outside the project working directory |
| `doom_loop` | Triggered when the same tool call repeats $3$ times with identical input |

`external_directory` and `doom_loop` default to `"ask"` — they act as safety trip-wires rather than regular tool controls. All other permissions default to `"allow"`, with one exception: `.env` files are denied for `read` by default to prevent accidental credential exposure.


**The `ask` interaction.** When OpenCode prompts for approval it offers three outcomes: `once` (approve this request only), `always` (approve future requests matching a suggested pattern for the rest of the session), or `reject` (deny). The `always` option is how a session incrementally builds up a set of trusted command patterns without needing them hard-coded upfront.


## Agents


OpenCode ships with two built-in **primary** agents and several built-in **subagents**:

| Agent | Type | Role |
| :-- | :--: | :-- |
| Build | primary | Default interactive agent; all tools enabled |
| Plan | primary | Restricted agent; prompts before file edits or bash commands |
| General | subagent | General-purpose agent for research and multi-step tasks |
| Explore | subagent | Fast read-only codebase exploration |

Primary agents are cycled with **Tab**. Subagents are invoked automatically by the orchestrator when a subtask matches their description, or manually via `@agent-name` in a message.


**Custom agents.** Each lives at `.opencode/agents/<name>.md`; the YAML frontmatter controls behavior and the body is the system prompt injected when that agent is active. The `notebook-writer` agent from this project, for instance:

```yaml
---
description: >
  Writes and reviews Jupyter notebook content matching the author's
  distinctive pedagogical style.
mode: subagent
temperature: 0.3
permission:
  edit: allow
  bash: deny
---
```

Key frontmatter fields:

| Field | Description |
| :-- | :-- |
| `description` | Shown in the agent picker; guides automatic routing by the orchestrator |
| `mode` | `subagent` (orchestrator-invoked), `primary` (Tab-switchable), or `all` |
| `temperature` | Sampling temperature (0.0–1.0); lower values yield more deterministic output |
| `steps` | Maximum agentic iterations before the agent is forced to respond with text |
| `model` | Override the model for this agent (e.g. use a cheaper model for Plan) |
| `permission` | Per-tool allow/ask/deny overrides; takes precedence over global config |
| `hidden` | If `true`, hides the agent from `@` autocomplete (useful for internal subagents) |

This project defines $4$ custom agents: `notebook-writer`, `notebook-planner`, `code-reviewer`, and `pytorch-expert`. Each carries a detailed system prompt with project-specific conventions.

:::{.callout-caution}
The `tools:` key in agent frontmatter is **deprecated**. Use `permission:` instead to control tool access. Mixing both in the same frontmatter can produce unexpected behavior.
:::


**Per-agent permission overrides.** Agent permissions are merged with the global config and take precedence. A common pattern is to lock down the global config conservatively and relax it per agent:

```json
{
  "permission": {
    "bash": { "*": "ask", "git *": "allow" }
  },
  "agent": {
    "build": {
      "permission": {
        "bash": {
          "*": "ask",
          "git *": "allow",
          "git commit *": "ask"
        }
      }
    }
  }
}
```

The same `permission:` block is valid in a `.md` frontmatter agent file — both JSON and markdown definitions use identical syntax.


**Default agent.** By default the Build agent opens on startup. We can change this with `default_agent` in `opencode.json`:

```json
{
  "default_agent": "plan"
}
```

The value must be a primary agent (not a subagent). If the specified agent doesn't exist or is a subagent, OpenCode falls back to `"build"` with a warning.


## Skills


**What skills are.** Skills are on-demand reference documents loaded into context via the `skill` tool. Unlike `AGENTS.md` and `instructions` files — which are always present in the context window — skills are loaded only when the agent determines they are relevant to the current task. This keeps routine sessions lean while making dense reference material available exactly when it is needed.


**Structure.** Each skill lives at `.opencode/skills/<name>/SKILL.md`. The frontmatter `name` and `description` fields are required; the body is the full reference content injected when the skill is invoked. Descriptions should be specific enough for the agent to route correctly.


**This repo's skills.** We maintain three skills:

| Skill | Description |
| :-- | :-- |
| `flet-dev` | Flet framework reference for v0.83+ (1.0 Beta track) |
| `matplotlib-style` | Plotting conventions extracted from the project's notebooks |
| `quarto-dev` | Quarto syntax and directives for notebook content |

Skills can run to $600$+ lines of dense reference material. Keeping them out of context until needed avoids bloating every session with irrelevant content — a framework-specific API reference is useless overhead during, say, a PyTorch training loop debugging session.

:::{.callout-note}
Skills are loaded automatically when the current task matches a skill's description. The active agent calls the `skill` tool, which injects the skill content into context for the duration of the task.
:::


## MCP Servers


The **Model Context Protocol** (MCP) is OpenCode's extensibility bus for external tools. Once an MCP server is configured, its tools are available to the LLM alongside built-in tools like `bash` and `edit` — no special invocation needed.

:::{.callout-caution}
Each MCP server adds its tool list to the context window. A large server like the GitHub MCP can easily push a session over the context limit. Only enable servers you actively use.
:::


**Local servers.** A local MCP server runs as a subprocess. We configure it with `type: "local"` and a `command` array:

```json
{
  "mcp": {
    "filesystem": {
      "type": "local",
      "command": ["npx", "-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
      "enabled": true
    }
  }
}
```

An optional `environment` object passes environment variables to the subprocess.


**Remote servers.** Remote MCP servers are reached over HTTP. We configure them with `type: "remote"` and a `url`:

```json
{
  "mcp": {
    "context7": {
      "type": "remote",
      "url": "https://mcp.context7.com/mcp"
    }
  }
}
```

For authenticated servers, we add a `headers` object — or use the `{env:VAR}` substitution to avoid committing credentials:

```json
{
  "mcp": {
    "context7": {
      "type": "remote",
      "url": "https://mcp.context7.com/mcp",
      "headers": {
        "CONTEXT7_API_KEY": "{env:CONTEXT7_API_KEY}"
      }
    }
  }
}
```

OpenCode also supports OAuth for remote MCP servers. When a server returns a 401, OpenCode initiates the OAuth flow automatically via Dynamic Client Registration (RFC 7591) and stores tokens in `~/.local/share/opencode/mcp-auth.json`. We can trigger this manually with `opencode mcp auth <server-name>`.


**Managing MCP tools.** MCP tools are registered under the server name as a prefix (e.g. `context7_resolve-library-id`). We can disable them globally or scope them to specific agents using the `tools` config with glob patterns:

```json
{
  "mcp": {
    "context7": { "type": "remote", "url": "https://mcp.context7.com/mcp" }
  },
  "tools": {
    "context7_*": false
  },
  "agent": {
    "notebook-writer": {
      "tools": { "context7_*": true }
    }
  }
}
```

This pattern — disable globally, enable per agent — keeps the default context clean while making the tool available exactly where it is useful.


**Example: Context7 for documentation search.** [Context7](https://github.com/upstash/context7) resolves library names to up-to-date documentation snippets. Adding it lets the agent answer questions about third-party APIs without hallucinating outdated method signatures:

```json
{
  "mcp": {
    "context7": {
      "type": "remote",
      "url": "https://mcp.context7.com/mcp"
    }
  }
}
```

We can either add `use context7` to a prompt explicitly, or add a line to `AGENTS.md`:

```markdown
When you need to look up library docs, use the `context7` tools.
```


## Config Deep-Dive


**Config locations and precedence.** OpenCode merges config from multiple sources in a defined order; later sources override earlier ones for conflicting keys, but non-conflicting settings accumulate:

| Priority | Source | Notes |
| :--: | :-- | :-- |
| 1 (lowest) | Remote (`.well-known/opencode`) | Organizational defaults |
| 2 | Global (`~/.config/opencode/opencode.json`) | User preferences, credentials |
| 3 | `OPENCODE_CONFIG` env var | Custom overrides |
| 4 | Project (`opencode.json`) | Project-specific settings |
| 5 | `.opencode/` directory | Agents, commands, plugins |
| 6 (highest) | `OPENCODE_CONFIG_CONTENT` env var | Runtime injection |

Because configs merge rather than replace, a project config that sets `model` will not wipe out the provider API key set in the global config.


**Variable substitution.** Two interpolation forms are available anywhere in config values:

- `{env:VAR_NAME}` — expands to the value of an environment variable (empty string if unset)
- `{file:path/to/file}` — expands to the contents of a file; path is relative to the config file or absolute

This is how we keep secrets out of version control:

```json
{
  "provider": {
    "anthropic": {
      "options": { "apiKey": "{env:ANTHROPIC_API_KEY}" }
    }
  }
}
```

And how we inject large prompt files by reference rather than inline:

```json
{
  "agent": {
    "build": { "prompt": "{file:./prompts/build.txt}" }
  }
}
```


**Model selection.** The global `model` key sets the primary model for all agents. The `small_model` key configures a separate lighter model for cheap background tasks like session title generation:

```json
{
  "model": "anthropic/claude-sonnet-4-5",
  "small_model": "anthropic/claude-haiku-4-5"
}
```

Per-agent model overrides are set via `agent.<name>.model`. If no per-agent model is specified, primary agents inherit the global model; subagents inherit the model of the primary agent that invoked them.


**Context compaction.** When a session approaches the model's context limit, OpenCode runs a compaction agent that summarizes the conversation and replaces older turns with the summary. This is on by default. The relevant config:

```json
{
  "compaction": {
    "auto": true,
    "prune": true,
    "reserved": 10000
  }
}
```

`prune` removes old tool outputs to reclaim tokens before falling back to full summarization. `reserved` is the token buffer held back from the context limit to give the compaction agent room to run without itself overflowing.


**Snapshots and undo.** OpenCode tracks all file changes via an internal git repository, which is what powers `/undo` and `/redo`. For large repositories with many submodules this can cause slow indexing and significant disk usage. We can disable it:

```json
{
  "snapshot": false
}
```

Disabling snapshots means the agent's file changes cannot be rolled back through the UI — useful on large monorepos where the cost outweighs the benefit, but only if the project is already under version control with a clean working tree before each session.


**Auto-update.** By default OpenCode downloads new versions on startup. We can disable this or switch to notify-only:

```json
{
  "autoupdate": false
}
```

Or `"autoupdate": "notify"` to see a banner without auto-installing. Note that auto-update has no effect when OpenCode was installed via a package manager like Homebrew — updates come through the package manager instead.


## Appendix: byobu for Multi-Pane Development


byobu is covered in detail in `runpod.ipynb`; we reproduce the keybinding reference here for convenience when working locally.

**Window management.**

| Key | Function |
| :--: | :-- |
| F2 | Create new window |
| F3 / F4 | Switch to previous / next window |
| F8 | Rename current window |
| F6 | Detach session |

**Pane management.**

| Key | Function |
| :--: | :-- |
| Ctrl+F2 | Create new vertical split |
| Shift+F2 | Create new horizontal split |
| Shift+F3 / F4 | Move focus to previous / next pane |
| Ctrl+F6 | Close current pane |

**Broadcast.** Running a command on all panes simultaneously:

| Key | Function |
| :--: | :-- |
| Shift+F9 | Toggle broadcast to all panes in window |

This is particularly useful when starting OpenCode sessions across multiple project directories in parallel — a single broadcast keystroke launches all of them at once.

:::{.callout-caution}
On macOS, F-keys may be bound to system actions (Exposé, Mission Control, etc.). Use `Fn+F<n>` to pass the raw key to byobu, or remap the conflicting shortcuts in **System Settings → Keyboard → Keyboard Shortcuts**.
:::


## Appendix: Terminal Productivity Tools


**`fzf`** is a general-purpose fuzzy finder for the terminal:

```{.bash filename="$ (local)"}
brew install fzf
```

Key integrations: `cd **<TAB>` triggers fuzzy directory navigation, and `Ctrl+R` replaces the default history search with an interactive fuzzy search over the full command history. OpenCode's `@`-file picker uses exactly this style of fuzzy matching, so familiarity with `fzf` transfers directly.


**`ripgrep`** (`rg`) is a fast recursive grep that respects `.gitignore` by default:

```{.bash filename="$ (local)"}
brew install ripgrep
```

OpenCode uses `rg` internally for codebase search — the Explore agent's file-content queries go through it. Basic usage:

```{.bash filename="$ (local)"}
rg "pattern" src/
```

This searches all files under `src/` recursively and reports file paths, line numbers, and matching lines.


---


■
